<a href="https://colab.research.google.com/github/JonasSantoos/cp_IOT/blob/main/atividade_ia_generativa.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Atividade: Assistente de IA Generativa com Hugging Face e Gemini

**Disciplina:** Disruptive Architectures: IoT, IoB & Generative AI

**Grupo:**
- Jonas Santos   RM563234
- Pedro Ferreira RM565824
- Gabriel Muller RM561995

**Tema escolhido:** Consultor de eficiência energética: dicas de consumo com base em medidores inteligentes e tomadas conectadas.

---

Neste notebook o grupo vai construir um assistente de IA em 4 etapas (e 1 bônus):

1. Assistente com personalidade (Hugging Face)
2. Comparação Hugging Face x Gemini
3. Chat com memória
4. Interface web com Gradio
5. (Bônus) API com FastAPI

Os trechos marcados com **`# >>> PERSONALIZE`** devem ser alterados pelo grupo.
Execute as células em ordem, de cima para baixo.

## 0. Configuração

In [ ]:
# huggingface_hub -> cliente para chamar modelos remotamente (API do Hugging Face)
# google-genai    -> SDK oficial do Google Gemini
# gradio          -> cria interfaces web a partir de funções Python
!pip install huggingface_hub google-genai gradio -q

In [ ]:
from huggingface_hub import InferenceClient
from google import genai
from google.genai import types
from google.colab import userdata

In [ ]:
# Os tokens ficam nos Secrets do Colab (ícone de chave na barra lateral)
HF_TOKEN = userdata.get("HF_TOKEN")
GEMINI_API_KEY = userdata.get("GEMINI_API_KEY")

print("HF_TOKEN ok" if HF_TOKEN else "ERRO: adicione HF_TOKEN nos Secrets do Colab")
print("GEMINI_API_KEY ok" if GEMINI_API_KEY else "ERRO: adicione GEMINI_API_KEY nos Secrets do Colab")

HF_TOKEN ok
GEMINI_API_KEY ok


In [ ]:
# Modelos usados na atividade (os mesmos da Aula 05)
MODELO_HF = "meta-llama/Llama-3.1-8B-Instruct"
MODELO_GEMINI = "gemini-3.5-flash-lite"

cliente_hf = InferenceClient(model=MODELO_HF, token=HF_TOKEN, provider="auto")
cliente_gemini = genai.Client(api_key=GEMINI_API_KEY)

In [ ]:
# >>> PERSONALIZE: descreva o assistente do grupo de acordo com o tema escolhido.
# Este é o "system": a instrução que define o comportamento do assistente.
# O exemplo abaixo é do tema 1 (casa inteligente). Troque pelo tema do grupo.

SYSTEM_PROMPT = """Você é um assistente de casa inteligente.
Ajude o usuário com dúvidas sobre automação residencial, sensores e dispositivos conectados.
Responda sempre em português, de forma clara, em no máximo 5 frases.
Se a pergunta não tiver relação com casa inteligente, diga educadamente que não pode ajudar."""

---
## Etapa 1: Assistente com personalidade (Hugging Face)

Aqui o grupo vai ver, na prática, o efeito do `system`.
A **mesma pergunta** será enviada com **instruções de sistema diferentes**.

In [ ]:
from google.genai import types

def perguntar_gemini(pergunta, system, temperatura=0.7):
    """Envia uma pergunta ao Gemini e devolve o texto da resposta."""
    resposta = cliente_gemini.models.generate_content(
        model=MODELO_GEMINI,
        contents=pergunta,
        config=types.GenerateContentConfig(
            system_instruction=system,
            temperature=temperatura,
            max_output_tokens=2000,
        ),
    )
    return resposta.text

In [ ]:
# >>> PERSONALIZE: crie 3 personalidades diferentes para o assistente do grupo.
personalidades = {
    "Consultor técnico": SYSTEM_PROMPT,
    "Coach de economia": (
        "Você é um coach motivacional de economia de energia. "
        "Fale de forma animada e próxima, dê desafios semanais práticos "
        "e comemore cada pequena economia do usuário. Responda em português."
    ),
    "Direto ao ponto": (
        "Você é um consultor de eficiência energética. "
        "Responda sempre em até 3 tópicos curtos, com uma dica prática por tópico. "
        "Sem introdução e sem conclusão. Responda em português."
    ),
}

# >>> PERSONALIZE: uma pergunta relacionada ao tema do grupo.
pergunta = "Como um sensor de presença pode ajudar a economizar energia em casa?"

for nome, system in personalidades.items():
    print(f"===== {nome} =====")
    print(perguntar_gemini(pergunta, system))   # trocado de perguntar_hf
    print()

===== Consultor técnico =====
Um sensor de presença ajuda a economizar energia ao acionar as luzes e aparelhos apenas quando detecta movimento em um ambiente. Assim, evita-se o desperdício de deixar cômodos iluminados ou climatizados sem ninguém. Além disso, você pode programá-lo para desligar os dispositivos automaticamente após um período de ausência. É uma solução prática que reduz significativamente o valor da conta de luz no final do mês.

===== Coach de economia =====
E aaaaaí, meu campeão ou campeã da economia! ⚡🚀 Que alegria ver você por aqui buscando formas inteligentes de cuidar do seu bolso e do nosso planeta! Adoro essa energia proativa!

Olha só, vamos falar de um verdadeiro **super-herói secreto** da sua casa: o **sensor de presença!** 🦸‍♂️💡

Sabe aquela velha mania de esquecer a luz acesa no corredor, na garagem, no banheiro de visitas ou na lavanderia? Pois é! O sensor de presença é aquele amigo implacável que não deixa nenhuma lâmpada acesa à toa. 

**Como ele funciona

**Observações do grupo (Etapa 1):**

- O que mudou nas respostas de cada personalidade?
- A pergunta foi a mesma e só o system mudou. O Consultor técnico respondeu num parágrafo objetivo. O Coach de economia foi animado, com emojis e texto longo, e acabou cortado no final pelo limite de tokens. O Direto ao ponto respondeu em 3 tópicos curtos, sem introdução nem conclusão.


- Qual `system` gerou a resposta mais útil para o tema? Por quê?
- O Direto ao ponto, porque deu dicas práticas e fáceis de aplicar, como o tempo do temporizador do sensor (1 a 3 minutos). O Consultor técnico ficou mais geral e o Coach gastou muito texto com empolgação.

---
## Etapa 2: Hugging Face x Gemini

Agora as mesmas perguntas vão para **dois modelos diferentes**.

No Gemini, o `system` não vai dentro da lista de mensagens. Ele é passado no parâmetro `system_instruction`.

In [ ]:
def perguntar_gemini(pergunta, system, modelo=None, temperatura=0.7):
    """Envia uma pergunta ao Gemini e devolve o texto da resposta."""
    resposta = cliente_gemini.models.generate_content(
        model=modelo or MODELO_GEMINI,
        contents=pergunta,
        config=types.GenerateContentConfig(
            system_instruction=system,
            temperature=temperatura,
            max_output_tokens=600,
        ),
    )
    return resposta.text

In [ ]:
MODELO_A = MODELO_GEMINI          # o modelo que você já usa (flash-lite)
MODELO_B = "gemini-3.5-flash"     # >>> confira se esse nome existe na sua conta

# >>> PERSONALIZE: 3 perguntas sobre o tema do grupo.
perguntas = [
    "Quais aparelhos da casa mais consomem energia?",
    "Qual a diferença entre uma tomada inteligente e um medidor de energia inteligente?",
    "Como interpretar o consumo em kWh na minha conta de luz?",
]

for p in perguntas:
    print("PERGUNTA:", p)
    print(f"\n--- {MODELO_A} ---")
    print(perguntar_gemini(p, SYSTEM_PROMPT, modelo=MODELO_A))
    print(f"\n--- {MODELO_B} ---")
    print(perguntar_gemini(p, SYSTEM_PROMPT, modelo=MODELO_B))
    print("\n" + "=" * 60 + "\n")

PERGUNTA: Quais aparelhos da casa mais consomem energia?

--- gemini-3.5-flash-lite ---
Na casa inteligente, os maiores consumidores de energia costumam ser os aparelhos que geram calor ou frio, como ar-condicionado, chuveiros elétricos e geladeiras. Para monitorar esse consumo em tempo real, você pode instalar medidores de energia inteligentes ou tomadas com Wi-Fi que medem a potência. Assim, fica mais fácil identificar os vilões da sua conta de luz e criar automações para desligá-los quando não estiverem em uso.

--- gemini-3.5-flash ---
Os aparelhos que mais consomem energia em uma casa geralmente são o ar-condicionado, o chuveiro elétrico, a geladeira e a secadora de roupas. Dispositivos que geram calor ou frio extremo costumam ser os maiores vilões da conta de luz. Com uma casa inteligente, você pode usar tomadas inteligentes com monitoramento de consumo para acompanhar o gasto de cada um em tempo real pelo celular. Além disso, é possível criar rotinas de automação para desligar e

# **Observações do grupo (Etapa 2):**

- Os dois modelos seguiram as regras do `SYSTEM_PROMPT` (idioma, tamanho, tema)?
- Sim. Os dois responderam em português, em um parágrafo curto e dentro do tema de eficiência energética e casa conectada. Só o gemini-3.5-flash teve respostas cortadas no final (nas perguntas 2 e 3) por causa do limite de tokens.


- Qual respondeu melhor? Em qual pergunta a diferença foi maior?
- O gemini-3.5-flash foi um pouco mais completo na primeira pergunta, citando também a secadora de roupas. Mas o gemini-3.5-flash-lite foi melhor no geral, porque respondeu tudo até o fim e explicou bem a diferença entre tomada e medidor inteligente. A diferença maior foi na segunda pergunta, em que o flash-lite deu a resposta completa e o flash foi cortado.


---
## Etapa 3: Chat com memória

O modelo **não guarda memória** entre uma chamada e outra.
Quem guarda a conversa é o nosso código, na lista `historico`, que é enviada inteira a cada mensagem.

Comandos do chat:
- `sair` encerra o chat
- `limpar` apaga o histórico (o assistente "esquece" a conversa)
- `historico` mostra quantas mensagens estão guardadas

**Teste sugerido:** diga seu nome, pergunte "qual é o meu nome?", digite `limpar` e pergunte de novo.

In [ ]:
historico = []

print("Chat iniciado! Comandos: sair | limpar | historico\n")

while True:
    entrada = input("Você: ")

    if entrada.lower() == "sair":
        print("Encerrando chat.")
        break

    if entrada.lower() == "limpar":
        historico = []  # apaga a conversa
        print("\n(histórico apagado)\n")
        continue

    if entrada.lower() == "historico":
        print(f"\n(mensagens no histórico: {len(historico)})\n")
        continue

    historico.append({"role": "user", "parts": [{"text": entrada}]})

    resposta = cliente_gemini.models.generate_content(
        model=MODELO_GEMINI,
        contents=historico,   # a conversa inteira é reenviada
        config=types.GenerateContentConfig(
            system_instruction=SYSTEM_PROMPT,
            max_output_tokens=600,
        ),
    )
    texto = resposta.text

    historico.append({"role": "model", "parts": [{"text": texto}]})

    print(f"\nAssistente: {texto}\n")

Chat iniciado! Comandos: sair | limpar | historico

Você: Jonas

Assistente: Olá, Jonas! Como posso ajudar você com a sua casa inteligente hoje? Você tem alguma dúvida sobre automação residencial, configuração de sensores ou dispositivos conectados? Estou à disposição para tornar sua casa mais prática e conectada!

Você: Qual é o meu nome?

Assistente: O seu nome é Jonas, conforme você me disse na nossa primeira mensagem. Como posso ajudar você com a sua casa inteligente hoje?

Você: historico

(mensagens no histórico: 4)

Você: limpar

(histórico apagado)

Você: historico

(mensagens no histórico: 0)

Você: sair
Encerrando chat.


In [ ]:
# Veja como ficou a lista enviada ao modelo
for msg in historico:
    texto = msg["parts"][0]["text"]
    print(f"[{msg['role']}] {texto[:80].replace(chr(10), ' ')}")

**Observações do grupo (Etapa 3):**

- O que aconteceu quando vocês perguntaram o nome antes e depois do `limpar`?
- Antes do limpar, a gente disse o nome (Jonas) e perguntou "Qual é o meu nome?". O assistente respondeu certo: "O seu nome é Jonas". Depois do limpar, o histórico voltou para 0 mensagens e o assistente não soube mais responder o nome.


- Explique, com suas palavras, por que isso acontece.
- O modelo não tem memória própria. Quem guarda a conversa é o nosso código, na lista historico, e essa lista inteira é enviada de novo a cada mensagem. Por isso, antes do limpar, o modelo "enxergava" o nome que a gente tinha dito. O limpar esvazia a lista, então a próxima chamada já não tem mais o nome e o modelo não tem como saber.

(responda aqui)

---
## Etapa 4: Interface web com Gradio

O assistente ganha uma interface web, com a opção de escolher o modelo (Hugging Face ou Gemini).

O Gradio entrega o histórico da conversa já no formato de `role`/`content`.
A função `texto_da_mensagem` existe porque, dependendo da versão do Gradio, o conteúdo chega como texto simples ou como lista.

In [ ]:
import gradio as gr

MODELOS = {
    "Gemini Flash-Lite": MODELO_GEMINI,
    "Gemini Flash": "gemini-3.5-flash",
}

def texto_da_mensagem(conteudo):
    """Extrai o texto de uma mensagem do histórico do Gradio."""
    if isinstance(conteudo, str):
        return conteudo
    if isinstance(conteudo, list):
        return " ".join(item.get("text", "") for item in conteudo if isinstance(item, dict))
    return str(conteudo)


def responder(mensagem, historico_gradio, modelo, temperatura):
    conteudos = []
    for msg in historico_gradio:
        papel = "model" if msg["role"] == "assistant" else "user"
        conteudos.append({"role": papel, "parts": [{"text": texto_da_mensagem(msg["content"])}]})
    conteudos.append({"role": "user", "parts": [{"text": mensagem}]})

    resposta = cliente_gemini.models.generate_content(
        model=MODELOS[modelo],
        contents=conteudos,
        config=types.GenerateContentConfig(
            system_instruction=SYSTEM_PROMPT,
            temperature=temperatura,
            max_output_tokens=2000,
        ),
    )
    return resposta.text

In [ ]:
# >>> PERSONALIZE: título, descrição e exemplos de acordo com o tema do grupo.
gr.ChatInterface(
    fn=responder,
    additional_inputs=[
        gr.Radio(list(MODELOS.keys()), value="Gemini Flash-Lite", label="Modelo"),
        gr.Slider(0.0, 1.0, value=0.7, step=0.1, label="Temperatura"),
    ],
    title="Consultor de Eficiência Energética",
    description="Pergunte sobre consumo de energia, medidores inteligentes e tomadas conectadas.",
    examples=[
        ["Quais aparelhos mais consomem energia em casa?", "Gemini Flash-Lite", 0.2],
        ["Me dê ideias criativas para economizar energia com tomadas inteligentes.", "Gemini Flash", 0.9],
    ],
).launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://1adfb99235775a2fcc.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


**Observações do grupo (Etapa 4):**

- Tire um print da interface funcionando e coloque no README do repositório do grupo.



- Troque de modelo no meio da conversa. O assistente continuou lembrando do que foi dito? Por quê?
- Sim, o assistente continuou lembrando do que foi dito. Isso acontece porque quem guarda a conversa é a interface do Gradio, e não o modelo. A cada mensagem, a função responder pega o histórico inteiro e envia de novo para o modelo escolhido. Então, mesmo trocando do Flash-Lite para o Flash, o novo modelo recebe toda a conversa anterior e consegue continuar de onde parou.


> Para parar a interface, interrompa a célula (botão de parar do Colab).

---
## Etapa 5 (Bônus): API com FastAPI

Aqui o assistente vira uma **API REST**, como no final da Aula 05.
Qualquer frontend (site, app, dispositivo IoT) poderia chamar esse endpoint.

A célula abaixo cria o arquivo `app.py`.

In [ ]:
%%writefile app.py
import os
from fastapi import FastAPI
from pydantic import BaseModel
from google import genai
from google.genai import types

# A chave e o system prompt vêm de variáveis de ambiente (nunca escreva a chave no código)
client = genai.Client(api_key=os.environ["GEMINI_API_KEY"])
MODELO = os.environ.get("MODELO_GEMINI", "gemini-3.5-flash-lite")
SYSTEM_PROMPT = os.environ.get("SYSTEM_PROMPT", "Você é um assistente prestativo.")

app = FastAPI()

# Um histórico separado para cada session_id
sessoes = {}

class Pergunta(BaseModel):
    mensagem: str
    session_id: str = "padrao"

@app.post("/chat")
def chat(pergunta: Pergunta):
    historico = sessoes.setdefault(pergunta.session_id, [])
    historico.append({"role": "user", "parts": [{"text": pergunta.mensagem}]})

    resposta = client.models.generate_content(
        model=MODELO,
        contents=historico,
        config=types.GenerateContentConfig(
            system_instruction=SYSTEM_PROMPT,
            max_output_tokens=1000,
        ),
    )
    texto = resposta.text
    historico.append({"role": "model", "parts": [{"text": texto}]})
    return {"session_id": pergunta.session_id, "resposta": texto}

Writing app.py


In [ ]:
!pip install fastapi uvicorn -q

import os, subprocess, time
from google.colab import userdata

os.environ["GEMINI_API_KEY"] = userdata.get("GEMINI_API_KEY")
os.environ["MODELO_GEMINI"] = MODELO_GEMINI
os.environ["SYSTEM_PROMPT"] = SYSTEM_PROMPT

servidor = subprocess.Popen(["uvicorn", "app:app", "--port", "8000"])
time.sleep(5)
print("Servidor rodando em http://localhost:8000")

Servidor rodando em http://localhost:8000


In [ ]:
import requests

def enviar(sessao, texto):
    r = requests.post("http://localhost:8000/chat",
                      json={"mensagem": texto, "session_id": sessao})
    print(f"[{sessao}] {r.status_code} -> {r.json()['resposta']}\n")

enviar("A", "Meu nome é Jonas.")
enviar("A", "Qual é o meu nome?")   # deve saber
enviar("B", "Qual é o meu nome?")   # não deve saber

[A] 200 -> Olá, Jonas! Como posso ajudar você com a sua casa inteligente hoje? Você tem dúvidas sobre automação, configuração de sensores ou escolha de dispositivos conectados? Estou à disposição para tornar sua residência mais prática e segura.

[A] 200 -> Seu nome é Jonas, conforme você me disse anteriormente. Como posso ajudar você com a sua casa inteligente hoje?

[B] 200 -> Como um assistente de casa inteligente, eu não tenho acesso a informações pessoais como o seu nome. Estou aqui para ajudar você com automação residencial, configuração de sensores e controle de dispositivos conectados. Como posso auxiliar na sua casa hoje?



In [ ]:
servidor.terminate()
print("Servidor encerrado.")

Servidor encerrado.
